<a href="https://colab.research.google.com/github/KSharif/Python/blob/main/molecualr_boilogy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 54.4 MB/s eta 0:00:00


In [ ]:
!pip install stable-baselines3[extra]


In [ ]:
!pip install Box2D


  Using cached Box2D-2.3.10-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (573 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 11.0 MB/s eta 0:00:00


In [ ]:
!pip install stable-baselines3[extra] gymnasium shimmy


  Using cached Shimmy-2.0.0-py3-none-any.whl.metadata (3.5 kB)
Using cached Shimmy-2.0.0-py3-none-any.whl (30 kB)


In [ ]:
!pip install stable-baselines3[extra] gymnasium shimmy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.1/958.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 19.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
import gymnasium as gym
from gymnasium import spaces

# Load the DNA sequence dataset
file_path = '/content/splice.data'
data = []
with open(file_path, 'r') as file:
    for line in file:
        parts = line.strip().split(",")
        label = parts[0].strip()
        sequence = parts[2].strip()
        data.append((label, sequence))

df = pd.DataFrame(data, columns=['Label', 'Sequence'])

# Encode the labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Label'])

# One-hot encode the DNA sequences
unique_chars = sorted(set(''.join(df['Sequence'])))
sequences = [list(seq) for seq in df['Sequence']]
ohe = OneHotEncoder(categories=[unique_chars], sparse_output=False)
X_encoded = np.array([ohe.fit_transform(np.array(seq).reshape(-1, 1)).flatten() for seq in sequences])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# ------------------------------------------------------
# Feature Selection Environment
# ------------------------------------------------------
class FeatureSelectionEnv(gym.Env):
    def __init__(self, X, y):
        super(FeatureSelectionEnv, self).__init__()
        self.X = X
        self.y = y
        self.n_features = X.shape[1]
        self.state = np.zeros(self.n_features, dtype=np.float32)
        self.action_space = spaces.MultiBinary(self.n_features)  # MultiBinary for feature selection
        self.observation_space = spaces.Box(
            low=0, high=1, shape=(self.n_features,), dtype=np.float32
        )

    def step(self, action):
        self.state = action  # Update state based on action
        selected_features = np.where(self.state == 1)[0]

        if len(selected_features) == 0:
            # Penalize for selecting no features
            reward = -1
            terminated = True
            truncated = False
        else:
            # Use only selected features
            X_selected = self.X[:, selected_features]
            model = RandomForestClassifier(random_state=42)
            model.fit(X_selected, self.y)
            accuracy = accuracy_score(self.y, model.predict(X_selected))
            reward = accuracy
            terminated = False
            truncated = False

        return self.state, reward, terminated, truncated, {}

    def reset(self, seed=None, options=None):
      if seed is not None:
        np.random.seed(seed)
      self.state = np.zeros(self.n_features, dtype=np.float32)
      return self.state, {}


# Initialize the environment
env = DummyVecEnv([lambda: FeatureSelectionEnv(X_train, y_train)])

# Train the RL agent
feature_selection_model = PPO("MlpPolicy", env, verbose=1)
feature_selection_model.learn(total_timesteps=10000)

# Evaluate the RL agent
obs = env.reset()
for _ in range(100):
    action, _ = feature_selection_model.predict(obs)
    obs, reward, done, _ = env.step(action)
    if done:
        break

# Get the selected features
selected_features = np.where(obs[0] == 1)[0]
X_train_selected = X_train[:, selected_features]
X_test_selected = X_test[:, selected_features]

print(f"Selected Features: {selected_features}")

# ------------------------------------------------------
# Hyperparameter Tuning Environment
# ------------------------------------------------------
class HyperparameterTuningEnv(gym.Env):
    def __init__(self, X, y):
        super(HyperparameterTuningEnv, self).__init__()
        self.X = X
        self.y = y
        self.state = [10, 1, 0.1]  # Initial hyperparameters: [n_estimators, max_depth, learning_rate]
        self.action_space = spaces.Discrete(3)  # Number of hyperparameters
        self.observation_space = spaces.Box(low=0, high=100, shape=(3,), dtype=np.float32)

    def step(self, action):
        # Modify the corresponding hyperparameter
        if action == 0:
            self.state[0] = min(self.state[0] + 10, 100)
        elif action == 1:
            self.state[1] = min(self.state[1] + 1, 20)
        elif action == 2:
            self.state[2] = max(self.state[2] - 0.01, 0.01)

        model = RandomForestClassifier(n_estimators=int(self.state[0]), max_depth=int(self.state[1]), random_state=42)
        model.fit(self.X, self.y)
        accuracy = accuracy_score(self.y, model.predict(self.X))
        reward = accuracy
        done = self.state[0] >= 100

        return np.array(self.state, dtype=np.float32), reward, done, {}

    def reset(self, seed=None, options=None):
    # Handle the seed for reproducibility
      if seed is not None:
        np.random.seed(seed)
      self.state = [10, 1, 0.1]  # Reset the hyperparameters to their initial values
      return np.array(self.state, dtype=np.float32), {}


# Initialize the environment
tuning_env = DummyVecEnv([lambda: HyperparameterTuningEnv(X_train_selected, y_train)])

# Train the RL agent
hyperparameter_tuning_model = PPO("MlpPolicy", tuning_env, verbose=1)
hyperparameter_tuning_model.learn(total_timesteps=10000)

# Evaluate the RL agent
tuning_obs = tuning_env.reset()
for _ in range(100):
    tuning_action, _ = hyperparameter_tuning_model.predict(tuning_obs)
    tuning_obs, tuning_reward, tuning_done, _ = tuning_env.step(tuning_action)
    if tuning_done:
        break

print(f"Optimized Hyperparameters: {tuning_obs[0]}")

# Final Model Evaluation
final_model = RandomForestClassifier(n_estimators=int(tuning_obs[0][0]), max_depth=int(tuning_obs[0][1]), random_state=42)
final_model.fit(X_train_selected, y_train)
predictions = final_model.predict(X_test_selected)
print("Final Model Performance:")
print(classification_report(y_test, predictions, target_names=label_encoder.classes_))


Using cuda device


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


-----------------------------
| time/              |      |
|    fps             | 2    |
|    iterations      | 1    |
|    time_elapsed    | 957  |
|    total_timesteps | 2048 |
-----------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 2          |
|    iterations           | 2          |
|    time_elapsed         | 1904       |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.17252198 |
|    clip_fraction        | 0.73       |
|    clip_range           | 0.2        |
|    entropy_loss         | -333       |
|    explained_variance   | -0.173     |
|    learning_rate        | 0.0003     |
|    loss                 | 8.39       |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.0966    |
|    value_loss           | 66.4       |
----------------------------------------
-----------------------------------------
| time/   

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


ValueError: not enough values to unpack (expected 5, got 4)

S0methings new trying

In [ ]:
# Install required libraries
!pip install gym numpy pandas scikit-learn torch

# Import libraries
import numpy as np
import pandas as pd
import gym
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Load the dataset
file_path = "/content/splice.data"  # Updated file path
column_names = ["Class", "DNA_Sequence", "Label"]
data = pd.read_csv(file_path, header=None, names=column_names, skiprows=0, delimiter=r"\s+")

# Preprocess the data
def preprocess_data(data):
    # Map class labels to integers
    class_mapping = {"EI": 0, "IE": 1, "N": 2}
    data["Class"] = data["Class"].map(class_mapping)

    # Split DNA sequences into individual characters and ensure uniform length
    sequences = data["DNA_Sequence"].apply(lambda x: list(x)).tolist()

    # Enforce uniform sequence length by padding with 'N' or truncating
    sequence_lengths = [len(seq) for seq in sequences]
    common_length = max(sequence_lengths)  # or choose a specific length

    padded_sequences = []
    for seq in sequences:
        if len(seq) < common_length:
            # Pad with 'N' to reach common_length
            padded = seq + ['N'] * (common_length - len(seq))
        else:
            padded = seq[:common_length]
        padded_sequences.append(padded)
    sequences = padded_sequences

    # Flatten the list of sequences into a single list of characters
    flattened_sequences = [char for seq in sequences for char in seq]

    # Reshape into a 2D array for OneHotEncoder
    flattened_sequences = np.array(flattened_sequences).reshape(-1, 1)

    # One-hot encode DNA sequences
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoded_sequences = encoder.fit_transform(flattened_sequences)

    # Reshape encoded sequences back to (num_samples, sequence_length, num_features)
    encoded_sequences = encoded_sequences.reshape(len(sequences), common_length, -1)

    # Flatten the last two dimensions to create a 2D array (num_samples, sequence_length * num_features)
    encoded_sequences = encoded_sequences.reshape(len(sequences), -1)

    # Split data into features (X) and labels (y)
    X = encoded_sequences
    y = data["Class"].values

    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    return X_train, X_test, y_train, y_test, encoder

X_train, X_test, y_train, y_test, encoder = preprocess_data(data)

# Define the environment
class DNAEnvironment(gym.Env):
    def __init__(self, X, y, window_size=10):
        super(DNAEnvironment, self).__init__()
        self.X = X
        self.y = y
        self.window_size = window_size
        self.current_index = 0
        self.sequence_length = X.shape[1]  # Total features = sequence_length * num_nucleotide_features
        self.action_space = gym.spaces.Discrete(4)

        # Observation is a 1D array of size (window_size * num_nucleotide_features)
        self.observation_space = gym.spaces.Box(
            low=0, high=1,
            shape=(window_size * encoder.n_features_in_,),  # Flattened window
            dtype=np.float32
        )

    def reset(self):
        self.current_index = 0
        return self._get_state()

    def _get_state(self):
        end_idx = self.current_index + self.window_size
        if end_idx > self.sequence_length:
            end_idx = self.sequence_length
        # Return a flattened window
        return self.X[self.current_index:end_idx].flatten()

    def step(self, action):
        done = False
        reward = 0

        if action == 0:  # Move right
            self.current_index += 1
            if self.current_index >= self.sequence_length - self.window_size:
                done = True
        else:  # Classify
            predicted_class = action - 1
            true_class = self.y[self.current_index]  # Use the label for the current index
            reward = 1 if (predicted_class == true_class) else -1
            done = True

        next_state = self._get_state() if not done else None
        return next_state, reward, done, {}


# Define the RL agent
class DQNAgent:
    def __init__(self, state_size, action_size, hidden_size=64, learning_rate=0.001, gamma=0.99, epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.995):
        self.state_size = state_size
        self.action_size = action_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=10000)
        self.model = self._build_model()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    def _build_model(self):
        model = nn.Sequential(
            nn.Linear(self.state_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.action_size)
        )
        return model

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        state = torch.FloatTensor(state).unsqueeze(0)
        q_values = self.model(state)
        return torch.argmax(q_values).item()

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return

        minibatch = random.sample(self.memory, batch_size)

        states = torch.FloatTensor(np.array([t[0] for t in minibatch]))
        actions = torch.LongTensor(np.array([t[1] for t in minibatch]))
        rewards = torch.FloatTensor(np.array([t[2] for t in minibatch]))

        # Replace None `next_state` with zero array
        next_states = np.array([
            t[3] if t[3] is not None else np.zeros(self.state_size)
            for t in minibatch
        ])
        next_states = torch.FloatTensor(next_states)

        dones = torch.FloatTensor(np.array([t[4] for t in minibatch]))

        current_q = self.model(states).gather(1, actions.unsqueeze(1))
        next_q = self.model(next_states).detach().max(1)[0]
        target_q = rewards + (1 - dones) * self.gamma * next_q

        loss = F.mse_loss(current_q.squeeze(), target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay


# Hyperparameters
state_size = X_train.shape[1]  # Shape is (num_samples, flattened_features)
action_size = 4
batch_size = 32
n_episodes = 1000

# Initialize environment and agent
env = DNAEnvironment(X_train, y_train)
agent = DQNAgent(state_size, action_size)

# Training loop
for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

    agent.replay(batch_size)

    if episode % 100 == 0:
        print(f"Episode: {episode}, Total Reward: {total_reward}, Epsilon: {agent.epsilon}")

# Evaluation
def evaluate_agent(env, agent):
    state = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        state = next_state
        total_reward += reward

    return total_reward

test_env = DNAEnvironment(X_test, y_test)
test_reward = evaluate_agent(test_env, agent)
print(f"Test Reward: {test_reward}")

# Classification report
y_pred = []
for i in range(len(X_test)):
    state = X_test[i].flatten()
    action = agent.act(state)
    y_pred.append(action - 1)

print(classification_report(y_test, y_pred, target_names=["EI", "IE", "N"]))

Episode: 0, Total Reward: -1, Epsilon: 1.0


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (32,) + inhomogeneous part.

In [16]:
# Install required libraries
!pip install gym numpy pandas scikit-learn torch

# Import libraries
import numpy as np
import pandas as pd
import gym
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Load the dataset
file_path = "/content/splice.data"  # Update the file path as needed
column_names = ["Class", "DNA_Sequence", "Label"]
data = pd.read_csv(file_path, header=None, names=column_names, skiprows=0, delimiter=r"\s+")

# Verify input data
if data.empty:
    raise ValueError("The input dataset is empty. Please provide a valid dataset.")
# Load the dataset
file_path = "/content/splice.data"  # Update the file path as needed
column_names = ["Class", "DNA_Sequence", "Label"]
data = pd.read_csv(file_path, header=None, names=column_names, skiprows=0, delimiter=r"\s+")

# Debug: Print dataset info
print("Dataset Info:")
print(data.info())
print("Preview of Dataset:")
print(data.head())

# Handle missing values
data["Class"] = data["Class"].map({"EI": 0, "IE": 1, "N": 2})
data["Class"] = data["Class"].fillna(2).astype(int)  # Replace missing values in Class with 'N'
data["DNA_Sequence"] = data["DNA_Sequence"].fillna("")  # Replace missing DNA sequences with empty strings

# Validate dataset
if data.empty:
    raise ValueError("The input dataset is empty. Please check the file path and content.")

# Preprocess the data
def preprocess_data(data):
    # Check for empty data after preprocessing
    if data.empty:
        raise ValueError("The dataset is empty after preprocessing. Please check the input data.")

    # Split DNA sequences into individual characters and ensure uniform length
    sequences = data["DNA_Sequence"].apply(lambda x: list(x)).tolist()
    sequence_lengths = [len(seq) for seq in sequences]
    common_length = max(sequence_lengths) if sequence_lengths else 0

    if common_length == 0:
        raise ValueError("No valid DNA sequences found in the dataset.")

    padded_sequences = []
    for seq in sequences:
        if len(seq) < common_length:
            padded = seq + ['N'] * (common_length - len(seq))
        else:
            padded = seq[:common_length]
        padded_sequences.append(padded)
    sequences = padded_sequences

    # Flatten the list of sequences into a single list of characters
    flattened_sequences = [char for seq in sequences for char in seq]

    # Reshape into a 2D array for OneHotEncoder
    flattened_sequences = np.array(flattened_sequences).reshape(-1, 1)

    # One-hot encode DNA sequences
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoded_sequences = encoder.fit_transform(flattened_sequences)

    # Reshape encoded sequences back to (num_samples, sequence_length, num_features)
    encoded_sequences = encoded_sequences.reshape(len(sequences), common_length, -1)
    encoded_sequences = encoded_sequences.reshape(len(sequences), -1)

    X = encoded_sequences
    y = data["Class"].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test, encoder

X_train, X_test, y_train, y_test, encoder = preprocess_data(data)

# Ensure no NaN values in y_test
if np.any(pd.isna(y_test)):
    raise ValueError("y_test contains NaN values. Check the preprocessing steps.")


# Ensure no NaN values in y_test
if np.any(pd.isna(y_test)):
    raise ValueError("y_test contains NaN values. Check the preprocessing steps.")

# Define the environment
class DNAEnvironment(gym.Env):
    def __init__(self, X, y, window_size=10):
        super(DNAEnvironment, self).__init__()
        self.X = X
        self.y = y
        self.window_size = window_size
        self.current_index = 0
        self.sequence_length = X.shape[1]
        self.action_space = gym.spaces.Discrete(4)
        self.observation_space = gym.spaces.Box(
            low=0, high=1,
            shape=(self.sequence_length,),
            dtype=np.float32
        )

    def reset(self):
        self.current_index = 0
        return self._get_state()

    def _get_state(self):
        return self.X[self.current_index].flatten()

    def step(self, action):
        done = False
        reward = 0

        if action == 0:  # Move right
            self.current_index += 1
            if self.current_index >= len(self.X):
                done = True
        else:  # Classify
            predicted_class = action - 1
            true_class = self.y[self.current_index]
            reward = 1 if (predicted_class == true_class) else -1
            done = True

        next_state = self._get_state() if not done else np.zeros(self.sequence_length)
        return next_state, reward, done, {}

# Define the RL agent
class DQNAgent:
    def __init__(self, state_size, action_size, hidden_size=64, learning_rate=0.001, gamma=0.99, epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.995):
        self.state_size = state_size
        self.action_size = action_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=10000)
        self.model = self._build_model()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    def _build_model(self):
        model = nn.Sequential(
            nn.Linear(self.state_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.action_size)
        )
        return model

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        state = torch.FloatTensor(state).unsqueeze(0)
        q_values = self.model(state)
        return torch.argmax(q_values).item()

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        states = torch.FloatTensor(np.array([t[0] for t in minibatch]))
        actions = torch.LongTensor(np.array([t[1] for t in minibatch]))
        rewards = torch.FloatTensor(np.array([t[2] for t in minibatch]))
        next_states = torch.FloatTensor(np.array([
            t[3] if t[3] is not None else np.zeros(self.state_size)
            for t in minibatch
        ]))
        dones = torch.FloatTensor(np.array([t[4] for t in minibatch]))

        current_q = self.model(states).gather(1, actions.unsqueeze(1))
        next_q = self.model(next_states).detach().max(1)[0]
        target_q = rewards + (1 - dones) * self.gamma * next_q

        loss = F.mse_loss(current_q.squeeze(), target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# Hyperparameters
state_size = X_train.shape[1]
action_size = 4
batch_size = 32
n_episodes = 1000

# Initialize environment and agent
env = DNAEnvironment(X_train, y_train)
agent = DQNAgent(state_size, action_size)

# Training loop
for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

    agent.replay(batch_size)

    if episode % 100 == 0:
        print(f"Episode: {episode}, Total Reward: {total_reward}, Epsilon: {agent.epsilon}")

# Evaluation
test_env = DNAEnvironment(X_test, y_test)
test_reward = sum(agent.act(X_test[i].flatten()) - 1 == y_test[i] for i in range(len(X_test)))
print(f"Test Reward: {test_reward}")

# Adjust y_pred to only consider valid class predictions
y_pred = [
    agent.act(X_test[i].flatten()) - 1  # Map actions to class labels
    for i in range(len(X_test))
]

# Remove invalid predictions (e.g., -1 for "move right" actions)
valid_indices = [i for i, pred in enumerate(y_pred) if pred in [0, 1, 2]]
y_pred_filtered = [y_pred[i] for i in valid_indices]
y_test_filtered = [y_test[i] for i in valid_indices]

# Generate the classification report with valid predictions
print(classification_report(y_test_filtered, y_pred_filtered, target_names=["EI", "IE", "N"]))



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3190 entries, 0 to 3189
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Class         3190 non-null   object
 1   DNA_Sequence  3190 non-null   object
 2   Label         3190 non-null   object
dtypes: object(3)
memory usage: 74.9+ KB
None
Preview of Dataset:
  Class         DNA_Sequence  \
0   EI,    ATRINS-DONOR-521,   
1   EI,    ATRINS-DONOR-905,   
2   EI,    BABAPOE-DONOR-30,   
3   EI,   BABAPOE-DONOR-867,   
4   EI,  BABAPOE-DONOR-2817,   

                                               Label  
0  CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCC...  
1  AGACCCGCCGGGAGGCGGAGGACCTGCAGGGTGAGCCCCACCGCCC...  
2  GAGGTGAAGGACGTCCTTCCCCAGGAGCCGGTGAGAAGCGCAGTCG...  
3  GGGCTGCGTTGCTGGTCACATTCCTGGCAGGTATGGGGCGGGGCTT...  
4  GCTCAGCCCCCAGGTCACCCAGGAACTGACGTGAGTGTCCCCATCC...  
Episode: 0, Total Reward: -1, Epsilon: 1.0
Episode: 100, Total Reward: -1, Epsilo

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
# Install required libraries (Run this in Jupyter Notebook or Google Colab)
!pip install gym numpy pandas scikit-learn torch

# Import libraries
import numpy as np
import pandas as pd
import gym
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Load the dataset
file_path = "/content/splice.data"  # Update the file path as needed
column_names = ["Class", "DNA_Sequence", "Label"]
data = pd.read_csv(file_path, header=None, names=column_names, skiprows=0, delimiter=r"\s+")

# Debug: Print dataset info
print("Dataset Info:")
print(data.info())
print("Preview of Dataset:")
print(data.head())

# Fix Class Mapping
data["Class"] = data["Class"].str.strip(",")  # Remove any trailing commas
data["Class"] = data["Class"].map({"EI": 0, "IE": 1, "N": 2})

# Ensure all class labels are valid
if data["Class"].isnull().any():
    raise ValueError("Invalid class labels found. Check data preprocessing.")

# Check Class Distribution
print("Class Distribution in Dataset:")
print(data["Class"].value_counts())

# Preprocess the data
def preprocess_data(data):
    # Split DNA sequences into individual characters
    sequences = data["DNA_Sequence"].apply(lambda x: list(x)).tolist()
    sequence_lengths = [len(seq) for seq in sequences]
    common_length = max(sequence_lengths) if sequence_lengths else 0

    if common_length == 0:
        raise ValueError("No valid DNA sequences found in the dataset.")

    # Pad sequences to ensure uniform length
    padded_sequences = [seq + ['N'] * (common_length - len(seq)) if len(seq) < common_length else seq[:common_length] for seq in sequences]

    # Flatten sequences into a list of characters
    flattened_sequences = np.array([char for seq in padded_sequences for char in seq]).reshape(-1, 1)

    # One-hot encode DNA sequences
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoded_sequences = encoder.fit_transform(flattened_sequences)

    # Reshape back to (num_samples, sequence_length, num_features)
    encoded_sequences = encoded_sequences.reshape(len(sequences), common_length, -1)
    encoded_sequences = encoded_sequences.reshape(len(sequences), -1)

    X = encoded_sequences
    y = data["Class"].values

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    return X_train, X_test, y_train, y_test, encoder

X_train, X_test, y_train, y_test, encoder = preprocess_data(data)

# Ensure Test Set Contains All Classes
print("Class Distribution in Test Set:")
print(pd.Series(y_test).value_counts())

# Define the RL Environment
class DNAEnvironment(gym.Env):
    def __init__(self, X, y):
        super(DNAEnvironment, self).__init__()
        self.X = X
        self.y = y
        self.current_index = 0
        self.sequence_length = X.shape[1]
        self.action_space = gym.spaces.Discrete(3)  # 3 classes: EI, IE, N
        self.observation_space = gym.spaces.Box(low=0, high=1, shape=(self.sequence_length,), dtype=np.float32)

    def reset(self):
        self.current_index = 0
        return self._get_state()

    def _get_state(self):
        return self.X[self.current_index].flatten()

    def step(self, action):
        done = True
        reward = 1 if action == self.y[self.current_index] else -1  # Reward correct classification, penalize wrong
        return np.zeros(self.sequence_length), reward, done, {}

# Define the DQN Agent
class DQNAgent:
    def __init__(self, state_size, action_size, hidden_size=64, learning_rate=0.001, gamma=0.99, epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.99):
        self.state_size = state_size
        self.action_size = action_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=10000)
        self.model = self._build_model()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    def _build_model(self):
        model = nn.Sequential(
            nn.Linear(self.state_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.action_size)
        )
        return model

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        state = torch.FloatTensor(state).unsqueeze(0)
        q_values = self.model(state)
        return torch.argmax(q_values).item()

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        states = torch.FloatTensor(np.array([t[0] for t in minibatch]))
        actions = torch.LongTensor(np.array([t[1] for t in minibatch]))
        rewards = torch.FloatTensor(np.array([t[2] for t in minibatch]))
        dones = torch.FloatTensor(np.array([t[4] for t in minibatch]))

        current_q = self.model(states).gather(1, actions.unsqueeze(1))
        next_q = self.model(states).detach().max(1)[0]
        target_q = rewards + (1 - dones) * self.gamma * next_q

        loss = F.mse_loss(current_q.squeeze(), target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# Initialize Environment & Agent
state_size = X_train.shape[1]
action_size = 3
batch_size = 32
n_episodes = 1000

env = DNAEnvironment(X_train, y_train)
agent = DQNAgent(state_size, action_size)

# Train the Agent
for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

    agent.replay(batch_size)

    if episode % 100 == 0:
        print(f"Episode: {episode}, Total Reward: {total_reward}, Epsilon: {agent.epsilon}")

# Evaluate the Model
y_pred = [agent.act(X_test[i].flatten()) for i in range(len(X_test))]
print(classification_report(y_test, y_pred, target_names=["EI", "IE", "N"]))


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3190 entries, 0 to 3189
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Class         3190 non-null   object
 1   DNA_Sequence  3190 non-null   object
 2   Label         3190 non-null   object
dtypes: object(3)
memory usage: 74.9+ KB
None
Preview of Dataset:
  Class         DNA_Sequence  \
0   EI,    ATRINS-DONOR-521,   
1   EI,    ATRINS-DONOR-905,   
2   EI,    BABAPOE-DONOR-30,   
3   EI,   BABAPOE-DONOR-867,   
4   EI,  BABAPOE-DONOR-2817,   

                                               Label  
0  CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCC...  
1  AGACCCGCCGGGAGGCGGAGGACCTGCAGGGTGAGCCCCACCGCCC...  
2  GAGGTGAAGGACGTCCTTCCCCAGGAGCCGGTGAGAAGCGCAGTCG...  
3  GGGCTGCGTTGCTGGTCACATTCCTGGCAGGTATGGGGCGGGGCTT...  
4  GCTCAGCCCCCAGGTCACCCAGGAACTGACGTGAGTGTCCCCATCC...  
Class Distribution in Dataset:
Class
2    1655
1     768
0     767
Name: count, d

In [18]:
# Install required libraries (Run this in Jupyter Notebook or Google Colab)
!pip install gym numpy pandas scikit-learn torch

# Import libraries
import numpy as np
import pandas as pd
import gym
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Load the dataset
file_path = "/content/splice.data"  # Update the file path as needed
column_names = ["Class", "DNA_Sequence", "Label"]
data = pd.read_csv(file_path, header=None, names=column_names, skiprows=0, delimiter=r"\s+")

# Debug: Print dataset info
print("Dataset Info:")
print(data.info())
print("Preview of Dataset:")
print(data.head())

# Fix Class Mapping
data["Class"] = data["Class"].str.strip(",")  # Remove any trailing commas
data["Class"] = data["Class"].map({"EI": 0, "IE": 1, "N": 2})

# Ensure all class labels are valid
if data["Class"].isnull().any():
    raise ValueError("Invalid class labels found. Check data preprocessing.")

# Check Class Distribution
print("Class Distribution in Dataset:")
print(data["Class"].value_counts())

# Handle Class Imbalance
def balance_dataset(data):
    class_0 = data[data["Class"] == 0]
    class_1 = data[data["Class"] == 1]
    class_2 = data[data["Class"] == 2]

    # Resample the minority classes
    class_0_upsampled = resample(class_0, replace=True, n_samples=len(class_2), random_state=42)
    class_1_upsampled = resample(class_1, replace=True, n_samples=len(class_2), random_state=42)

    # Combine and shuffle the dataset
    balanced_data = pd.concat([class_0_upsampled, class_1_upsampled, class_2])
    balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)
    return balanced_data

data_balanced = balance_dataset(data)

# Preprocess the data
def preprocess_data(data):
    # Split DNA sequences into individual characters
    sequences = data["DNA_Sequence"].apply(lambda x: list(x)).tolist()
    sequence_lengths = [len(seq) for seq in sequences]
    common_length = max(sequence_lengths) if sequence_lengths else 0

    if common_length == 0:
        raise ValueError("No valid DNA sequences found in the dataset.")

    # Pad sequences to ensure uniform length
    padded_sequences = [seq + ['N'] * (common_length - len(seq)) if len(seq) < common_length else seq[:common_length] for seq in sequences]

    # Flatten sequences into a list of characters
    flattened_sequences = np.array([char for seq in padded_sequences for char in seq]).reshape(-1, 1)

    # One-hot encode DNA sequences
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoded_sequences = encoder.fit_transform(flattened_sequences)

    # Reshape back to (num_samples, sequence_length, num_features)
    encoded_sequences = encoded_sequences.reshape(len(sequences), common_length, -1)
    encoded_sequences = encoded_sequences.reshape(len(sequences), -1)

    X = encoded_sequences
    y = data["Class"].values

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    return X_train, X_test, y_train, y_test, encoder

X_train, X_test, y_train, y_test, encoder = preprocess_data(data_balanced)

# Ensure Test Set Contains All Classes
print("Class Distribution in Test Set:")
print(pd.Series(y_test).value_counts())

# Define the RL Environment
class DNAEnvironment(gym.Env):
    def __init__(self, X, y):
        super(DNAEnvironment, self).__init__()
        self.X = X
        self.y = y
        self.current_index = 0
        self.sequence_length = X.shape[1]
        self.action_space = gym.spaces.Discrete(3)  # 3 classes: EI, IE, N
        self.observation_space = gym.spaces.Box(low=0, high=1, shape=(self.sequence_length,), dtype=np.float32)

    def reset(self):
        self.current_index = 0
        return self._get_state()

    def _get_state(self):
        return self.X[self.current_index].flatten()

    def step(self, action):
        done = True
        if action == self.y[self.current_index]:  # Correct classification
            reward = 2 if self.y[self.current_index] in [0, 1] else 1  # Higher reward for minority classes
        else:  # Incorrect classification
            reward = -2 if self.y[self.current_index] in [0, 1] else -1  # Higher penalty for minority classes
        return np.zeros(self.sequence_length), reward, done, {}

# Define the DQN Agent
class DQNAgent:
    def __init__(self, state_size, action_size, hidden_size=64, learning_rate=0.001, gamma=0.99, epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.99):
        self.state_size = state_size
        self.action_size = action_size
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.memory = deque(maxlen=10000)
        self.model = self._build_model()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    def _build_model(self):
        model = nn.Sequential(
            nn.Linear(self.state_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.ReLU(),
            nn.Linear(self.hidden_size, self.action_size)
        )
        return model

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        state = torch.FloatTensor(state).unsqueeze(0)
        q_values = self.model(state)
        return torch.argmax(q_values).item()

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        states = torch.FloatTensor(np.array([t[0] for t in minibatch]))
        actions = torch.LongTensor(np.array([t[1] for t in minibatch]))
        rewards = torch.FloatTensor(np.array([t[2] for t in minibatch]))
        dones = torch.FloatTensor(np.array([t[4] for t in minibatch]))

        current_q = self.model(states).gather(1, actions.unsqueeze(1))
        next_q = self.model(states).detach().max(1)[0]
        target_q = rewards + (1 - dones) * self.gamma * next_q

        loss = F.mse_loss(current_q.squeeze(), target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# Initialize Environment & Agent
state_size = X_train.shape[1]
action_size = 3
batch_size = 32
n_episodes = 2000  # Extended training episodes

env = DNAEnvironment(X_train, y_train)
agent = DQNAgent(state_size, action_size)

# Train the Agent
for episode in range(n_episodes):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

    agent.replay(batch_size)

    if episode % 100 == 0:
        print(f"Episode: {episode}, Total Reward: {total_reward}, Epsilon: {agent.epsilon}")

# Evaluate the Model
y_pred = [agent.act(X_test[i].flatten()) for i in range(len(X_test))]
print(classification_report(y_test, y_pred, target_names=["EI", "IE", "N"]))


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3190 entries, 0 to 3189
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Class         3190 non-null   object
 1   DNA_Sequence  3190 non-null   object
 2   Label         3190 non-null   object
dtypes: object(3)
memory usage: 74.9+ KB
None
Preview of Dataset:
  Class         DNA_Sequence  \
0   EI,    ATRINS-DONOR-521,   
1   EI,    ATRINS-DONOR-905,   
2   EI,    BABAPOE-DONOR-30,   
3   EI,   BABAPOE-DONOR-867,   
4   EI,  BABAPOE-DONOR-2817,   

                                               Label  
0  CCAGCTGCATCACAGGAGGCCAGCGAGCAGGTCTGTTCCAAGGGCC...  
1  AGACCCGCCGGGAGGCGGAGGACCTGCAGGGTGAGCCCCACCGCCC...  
2  GAGGTGAAGGACGTCCTTCCCCAGGAGCCGGTGAGAAGCGCAGTCG...  
3  GGGCTGCGTTGCTGGTCACATTCCTGGCAGGTATGGGGCGGGGCTT...  
4  GCTCAGCCCCCAGGTCACCCAGGAACTGACGTGAGTGTCCCCATCC...  
Class Distribution in Dataset:
Class
2    1655
1     768
0     767
Name: count, d

In [2]:
!pip install stable-baselines3[extra] gymnasium numpy pandas scikit-learn

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
import gymnasium as gym
from gymnasium import spaces

# Load the DNA sequence dataset
file_path = '/content/splice.data'
data = []
with open(file_path, 'r') as file:
    for line in file:
        parts = line.strip().split(",")
        label = parts[0].strip()
        sequence = parts[2].strip()
        data.append((label, sequence))

df = pd.DataFrame(data, columns=['Label', 'Sequence'])

# Encode the labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Label'])

# One-hot encode the DNA sequences
unique_chars = sorted(set(''.join(df['Sequence'])))
sequences = [list(seq) for seq in df['Sequence']]
ohe = OneHotEncoder(categories=[unique_chars], sparse_output=False)
X_encoded = np.array([ohe.fit_transform(np.array(seq).reshape(-1, 1)).flatten() for seq in sequences])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# ------------------------------------------------------
# Stage 1: Sequence Classification with RL
# ------------------------------------------------------
class SequenceClassificationEnv(gym.Env):
    def __init__(self, X, y):
        super(SequenceClassificationEnv, self).__init__()
        self.X = X
        self.y = y
        self.current_index = 0
        self.n_classes = len(np.unique(y))
        self.action_space = spaces.Discrete(self.n_classes)  # Actions: Predict one of the classes
        self.observation_space = spaces.Box(
            low=0, high=1, shape=(self.X.shape[1],), dtype=np.float32
        )

    def step(self, action):
        reward = 1 if action == self.y[self.current_index] else -1
        self.current_index += 1
        done = self.current_index >= len(self.y)

        obs = self.X[self.current_index - 1] if not done else np.zeros_like(self.X[0])

        return obs, reward, done, False, {}  # Ensure you return five values


    def reset(self, seed=None, options=None):
        self.current_index = 0
        if seed is not None:
            np.random.seed(seed)  # Set the seed for reproducibility
        return self.X[self.current_index], {}  # Return the observation and an empty info dictionary


# Initialize the environment for sequence classification
seq_env = DummyVecEnv([lambda: SequenceClassificationEnv(X_train, y_train)])

# Train the RL agent for sequence classification
seq_model = PPO("MlpPolicy", seq_env, verbose=1)
seq_model.learn(total_timesteps=10000)

# ------------------------------------------------------
# Stage 2: Feature Selection with RL
# ------------------------------------------------------
class FeatureSelectionEnv(gym.Env):
    def __init__(self, X, y):
        super(FeatureSelectionEnv, self).__init__()
        self.X = X
        self.y = y
        self.n_features = X.shape[1]
        self.state = np.zeros(self.n_features, dtype=np.float32)
        self.action_space = spaces.MultiBinary(self.n_features)  # Actions: Select or not select features
        self.observation_space = spaces.Box(
            low=0, high=1, shape=(self.n_features,), dtype=np.float32
        )

    def step(self, action):
        self.state = action
        selected_features = np.where(self.state == 1)[0]

        if len(selected_features) == 0:
            reward = -1
            done = True
        else:
            X_selected = self.X[:, selected_features]
            model = RandomForestClassifier(random_state=42)
            model.fit(X_selected, self.y)
            accuracy = accuracy_score(self.y, model.predict(X_selected))
            reward = accuracy
            done = False

        return self.state, reward, done, False, {}  # Return five values

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)  # Set the seed for reproducibility
        self.state = np.zeros(self.n_features, dtype=np.float32)
        return self.state, {}  # Return the state and an empty info dictionary




# Initialize the environment for feature selection
feat_env = DummyVecEnv([lambda: FeatureSelectionEnv(X_train, y_train)])

# Train the RL agent for feature selection
feat_model = PPO("MlpPolicy", feat_env, verbose=1)
feat_model.learn(total_timesteps=10000)

# Get the selected features
selected_features = np.where(feat_model.predict(feat_env.reset()[0])[0] == 1)[0]
X_train_selected = X_train[:, selected_features]
X_test_selected = X_test[:, selected_features]

# ------------------------------------------------------
# Stage 3: Hyperparameter Tuning with RL
# ------------------------------------------------------
class HyperparameterTuningEnv(gym.Env):
    def __init__(self, X, y):
        super(HyperparameterTuningEnv, self).__init__()
        self.X = X
        self.y = y
        self.state = [10, 1]  # Hyperparameters: [n_estimators, max_depth]
        self.action_space = spaces.Discrete(2)  # Actions: Adjust hyperparameters
        self.observation_space = spaces.Box(low=0, high=100, shape=(2,), dtype=np.float32)

    def step(self, action):
        if action == 0:
            self.state[0] = min(self.state[0] + 10, 100)  # Increase n_estimators
        elif action == 1:
            self.state[1] = min(self.state[1] + 1, 20)  # Increase max_depth

        model = RandomForestClassifier(n_estimators=int(self.state[0]), max_depth=int(self.state[1]), random_state=42)
        model.fit(self.X, self.y)
        accuracy = accuracy_score(self.y, model.predict(self.X))
        reward = accuracy
        done = self.state[0] >= 100

        return np.array(self.state, dtype=np.float32), reward, done, False, {}  # Return five values

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)  # Set the seed for reproducibility
        self.state = [10, 1]  # Reset the hyperparameters
        return np.array(self.state, dtype=np.float32), {}  # Return the state and an empty info dictionary



# Initialize the environment for hyperparameter tuning
tune_env = DummyVecEnv([lambda: HyperparameterTuningEnv(X_train_selected, y_train)])

# Train the RL agent for hyperparameter tuning
tune_model = PPO("MlpPolicy", tune_env, verbose=1)
tune_model.learn(total_timesteps=10000)

# ------------------------------------------------------
# Final Model Evaluation
# ------------------------------------------------------
# Train a final model with the selected features and optimized hyperparameters
final_hyperparameters = tune_model.predict(tune_env.reset()[0])[0]
final_model = RandomForestClassifier(
    n_estimators=int(final_hyperparameters[0]),
    max_depth=int(final_hyperparameters[1]),
    random_state=42
)
final_model.fit(X_train_selected, y_train)
predictions = final_model.predict(X_test_selected)

# Print final evaluation results
print("Final Model Performance:")
print(classification_report(y_test, predictions, target_names=label_encoder.classes_))


Using cuda device


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


-----------------------------
| time/              |      |
|    fps             | 451  |
|    iterations      | 1    |
|    time_elapsed    | 4    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 450         |
|    iterations           | 2           |
|    time_elapsed         | 9           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.012019414 |
|    clip_fraction        | 0.121       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -0.0125     |
|    learning_rate        | 0.0003      |
|    loss                 | 3.39        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0248     |
|    value_loss           | 9.84        |
-----------------------------------------
----------------------------------

/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


-----------------------------
| time/              |      |
|    fps             | 5    |
|    iterations      | 1    |
|    time_elapsed    | 344  |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 5           |
|    iterations           | 2           |
|    time_elapsed         | 750         |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.020557843 |
|    clip_fraction        | 0.452       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.674      |
|    explained_variance   | -0.0418     |
|    learning_rate        | 0.0003      |
|    loss                 | 2.48        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.049      |
|    value_loss           | 11.4        |
-----------------------------------------
----------------------------------